<a href="https://colab.research.google.com/github/anonymous-capybara/platune-mini/blob/main/train-mini-audio-latent-diffusion-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Training an Audio Latent Diffusion Transformer (DiT)  

**(Colab version)**

This notebook uses the [PLaTune-mini](#) repository to train an audio generative model on a small guitar dataset: [GuitarSet](https://zenodo.org/records/3371780).

PLaTune (Pretrained Latents Tuner) (Nabi et al., 2025) is a Latent Diffusion Transformer for generating audio data with the option to add musical controls (e.g., melody, dynamics, instrument types). It implements a technique for latent diffusion called [Rectified Flow](https://rectifiedflow.github.io/),

PLaTune-mini is a toy version of PLaTune, with a smaller number of model parameters, and an additional data augmentation technique to allow models to **train on smaller datasets**.



**Original research paper of PLaTune:**  
> Nabi, S., Demerlé, N., Peeters, G., Bevilacqua, F., Esling, P., 2025. Adding temporal musical controls on top of pretrained generative models, in: Proceedings of the 26th International Society for Music Information Retrieval Conference. ISMIR, pp. 793–800. https://doi.org/10.5281/zenodo.17811482

## 1. Config you environment

<!-- **If you're on your own device**:
 - Make sure to create a **new conda environment** (!!), using the following command:  
    - `conda create -n platune python=3.11`  
 - We will be using this new `platune` env for this notebook (not the `emi` that we created for the unit)
 - Make sure to select the `platune` env on the top-right corner of VS Code -->

**If you're on Colab**:
 - You're good to go, please run the config below every time you start a new Colab session.

In steps below, we'll download the [platune-mini](https://github.com/anonymous-capybara/platune-mini) repository, a small subset of [GuitarSet](https://zenodo.org/records/3371780) dataset, and install required libraries.  



In [ ]:
!git clone https://github.com/anonymous-capybara/platune-mini.git

Download an unzip the GuitarSet dataset (656.9 MB) (this can take a while)

In [ ]:
import urllib.request
import zipfile
import os

url = "https://zenodo.org/records/3371780/files/audio_mono-mic.zip"
zip_path = "audio_mono-mic.zip"
out_dir = "audio_mono-mic"

print("Downloading...")
urllib.request.urlretrieve(url, zip_path)

print("Extracting...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(out_dir)

os.remove(zip_path)
print("Done.")

In [ ]:
import sys, os

venv_name = os.path.basename(sys.prefix)
if venv_name == "emi" or venv_name == "base":
    print(f"Warning: You are currently in the '{venv_name}' virtual environment. It is recommended to create a new virtual environment for this project to avoid dependency conflicts.")
    sys.exit(1)
else:
    print(f"You are currently in the '{venv_name}' virtual environment. Proceeding with the setup.")


!pip install -r platune-mini/requirements.txt

if sys.platform == 'darwin':
    # MacOS
    print("Installing PyTorch for MacOS...")
    !conda install -c conda-forge libsndfile -y
    !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
    !pip install tensorboard==2.20.0 setuptools==81.0.0 --force-reinstall
else:
    # Windows/Linux with CUDA
    print("Installing PyTorch for CUDA...")
    !pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
    !pip install tensorboard>=2.20.0 setuptools==81.0.0 --force-reinstall

## 2. Prepare Dataset

### 2.1 Extract latents from Guitarset, using a pretrained autoencoder  

Latent diffusion models operate in the latent space of autoencoders, therefore the first step is to create a latent dataset by running the dataset through a pretrained autoencoder. We'll use a script for this.  

**Note:** The extracted dataset will be save to the folder defined by the `--output_path` argument below, and will be used later for training. Therefore if you're restarting from a checkpoint, you don't need to do this step again.

In [ ]:
%cd platune-mini

In [ ]:
!python prepare_dataset.py \
    --output_path "training_data/guitatset" \
    --input_path "../audio_mono-mic" \
    --config simple \
    --parser_name simple_parser \
    --emb_model_path music2latent \
    --gpu 0 \
    --db_size 2 \
    --num_signal 131072 \
    --cut_silences \
    --save_waveform \
    -l rms -l integrated_loudness -l loudness1s \
    --val_num_chunks 100

## 3. Train PLaTune-mini

### 3.1 Import the training pieces

This cell imports the mini model, the dataset loader, and a couple of small helper functions. The helpers are only here to keep the actual training cells easy to read.

In [ ]:
%cd platune-mini

In [ ]:
import json
import os
import pytorch_lightning as pl
import torch

from platune.datasets.dataset import load_data
from platune.model_mini import PLaTuneMini

torch.set_float32_matmul_precision("high")
from platune.utils import search_for_run, select_accelerator


### 3.2 Choose the run settings

This is the main cell you are likely to edit. It defines where the training and validation LMDBs live, where checkpoints will be saved, and a few core training hyperparameters.

In [ ]:
db_path = "training_data/guitatset"
val_lmdb_path = "training_data/guitatset_val"
name = "guitatset_mini"
save_path = "runs"

gpu = 0
ckpt = None  # e.g. "runs/guitatset_mini/version_0/checkpoints/last.ckpt"

max_steps = 300_000
val_every = 15_000
lmdb_keys_filename = "lmdb_keys"

batch_size = 32
n_workers = 7
lr = 1e-4
nb_steps = 20
n_audio_examples = 12


### 3.3 Load the unconditional latent dataset

PLaTune-mini is unconditional, so we pass empty control lists into the existing dataset loader. The loader still returns the same tuple shape as the full model pipeline, but the mini model only uses the latent tensor `z`.

In [ ]:
train_loader, val_loader, _ = load_data(
    data_path=db_path,
    discrete_keys=[],
    continuous_keys=[],
    batch_size=batch_size,
    n_workers=n_workers,
    lmdb_keys_file=lmdb_keys_filename,
    val_data_path=val_lmdb_path,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


#### Visualise some latents in the dataset

In [ ]:
import matplotlib.pyplot as plt

batch = next(iter(train_loader))

z = batch[0].detach().cpu()

print(f"latent batch shape: {tuple(z.shape)}")
print(f"min: {z.min().item():.4f}, max: {z.max().item():.4f}")

# Visualize the first sample in the batch
sample = z[0]  # shape: (latent_dim, time)
sample_2d = sample.reshape(sample.shape[0], -1)
plt.figure(figsize=(6, 3))
plt.imshow(sample_2d.numpy(), aspect="auto", origin="lower", cmap="magma")
plt.xlabel("Time")
plt.ylabel("Latent Dimension")

plt.tight_layout()
plt.show()

### 3.4. Build the mini model and save the run config

Here we start from the default PLaTune-mini settings, apply a few notebook overrides, and save them to `config.json` so the run can be reproduced later.  

In [ ]:
model_config = PLaTuneMini.default_model_config()
model_config.update({
    "lr": lr,
    "nb_steps": nb_steps,
    "n_audio_examples": n_audio_examples,
})
model = PLaTuneMini(**model_config)

run_dir = os.path.join(save_path, name)
os.makedirs(run_dir, exist_ok=True)

config = {
    "cli_args": {
        "db_path": db_path,
        "val_lmdb_path": val_lmdb_path,
        "name": name,
        "save_path": save_path,
        "gpu": gpu,
        "ckpt": ckpt,
        "max_steps": max_steps,
        "val_every": val_every,
        "lmdb_keys_filename": lmdb_keys_filename,
        "batch_size": batch_size,
        "n_workers": n_workers,
        "lr": lr,
        "nb_steps": nb_steps,
        "n_audio_examples": n_audio_examples,
    },
    "model_config": model_config,
}

with open(os.path.join(run_dir, "config.json"), "w") as config_out:
    json.dump(config, config_out, indent=2)

config


### 3.5 Create the Lightning trainer

This cell sets up checkpointing, picks the device, and decides how often validation should run. Validation logs synthesized audio samples so you can hear how the diffusion model evolves during training.

In [ ]:
callbacks = [pl.callbacks.ModelCheckpoint(filename="last")]

val_check = {}
if val_loader is not None and len(train_loader) > 0:
    if len(train_loader) >= val_every:
        val_check["val_check_interval"] = val_every
        print(f"Validation will be checked every {val_every} training steps.")
    else:
        n_epoch = max(1, val_every // len(train_loader))
        val_check["check_val_every_n_epoch"] = n_epoch
        print(f"Validation will be checked every {n_epoch} epochs.")

accelerator, devices = select_accelerator(gpu)
print(f"device - selected gpu: {accelerator}:{devices}")

trainer = pl.Trainer(
    logger=pl.loggers.TensorBoardLogger(save_path, name=name),
    accelerator=accelerator,
    devices=devices,
    callbacks=callbacks,
    max_epochs=100000,
    max_steps=max_steps,
    profiler="simple",
    enable_progress_bar=True,
    **val_check,
)

### 3.6 (Colab) Monitor the training via TensorBoard  

We can use the TensorBoard library to monitor the training loss, and display some examples generated during the process.   

The code below should open a TensorBoard panel in its output area.

**Monitor your training progress here**

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir /content/platune-mini/runs

### 3.7 Start training

If `ckpt` points to a previous checkpoint, training resumes from there. Otherwise this starts a fresh unconditional PLaTune-mini run.

In [ ]:
run = search_for_run(ckpt)
if run is not None:
    print(f"Restarting from checkpoint: {run}")

trainer.fit(model, train_loader, val_loader, ckpt_path=run)


**Note:** This is typically a very long training process (e.g., 15 ~ 30 mins at minimal on small datasets, or several hours / days for larger dataset). Therefore, checkpointing is very important, **go back to Step 3.6 to monitor the progress**.

## 4. Generate some samples

After training, we can draw fresh random latent states `s`, map them into audio latents `z` with the diffusion model, and decode them back to waveform audio.


In [ ]:
from IPython.display import Audio, Markdown, display

# In the unconditional mini model, a random latent state acts like a random style seed.
n_generate = 4
sample_nb_steps = nb_steps
sample_seed = 1234

In [ ]:
sample_model = model.eval()
device = next(sample_model.flow.parameters()).device

reference_batch = next(iter(val_loader))
reference_z = reference_batch[0]
sample_frames = reference_z.shape[-1]

torch.manual_seed(sample_seed)

with torch.no_grad():
    s = torch.randn(n_generate, sample_model.latent_dim, sample_frames, device=device)
    z_samples = sample_model.s_to_z(s, nb_steps=sample_nb_steps)
    audio_samples = sample_model.decode_latent_to_audio(z_samples).detach().cpu()

audio_samples = audio_samples / audio_samples.abs().amax(dim=1, keepdim=True).clamp_min(1e-6)

print(f"Generated {audio_samples.shape[0]} samples")
print(f"Audio shape: {tuple(audio_samples.shape)}")


In [ ]:
for i, audio in enumerate(audio_samples):
    display(Markdown(f"**Random sample {i + 1}**"))
    display(Audio(audio.numpy(), rate=sample_model.sample_rate))
